# FIFA World Cup ML Analytics: Match Prediction and Player Clustering

This notebook walks through an introductory machine learning project using player-level FIFA World Cup performance data.

The project has two parts:

1. Supervised learning: predict whether Team A wins, draws, or loses against Team B.
2. Unsupervised learning: cluster players by performance style and overperformance.

## 1. Setup

The reusable functions live in `main.py`. This keeps the notebook beginner-friendly while making the full project reproducible from a script.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import main

print(PROJECT_ROOT)

## 2. Load and Inspect the Dataset

The raw dataset is player-level. Each row represents one player's performance in one match. The target result is match-level, so we need to transform the data before training the supervised model.

In [ ]:
raw_df = main.load_dataset(main.DATA_FILE)
inspection = main.inspect_dataset(raw_df)
raw_df.head()

The inspection step prints shape, columns, data types, missing values, team count, player count, match count, and value counts for useful categorical columns when they exist.

## 3. Data Cleaning

The cleaning step removes exact duplicate rows, converts numeric-looking columns to numeric types, replaces infinite values, and fills missing values. This keeps the workflow simple and safe for beginners.

In [ ]:
df = main.clean_data(raw_df)
df.isna().sum().sum()

## 4. Exploratory Data Analysis

EDA charts help us understand the dataset before modeling. The workflow saves charts for result distribution, position trends, team trends, expected versus actual output, and correlations.

In [ ]:
main.create_eda_charts(df)
sorted((main.FIGURES_DIR).glob('*.png'))[:5]

## 5. Feature Selection

The supervised model uses player performance columns that exist in the dataset. The workflow automatically skips missing columns.

`performance_score` and `tournament_rating` are excluded because they may leak result information or summarize performance too directly.

In [ ]:
supervised_columns = main.select_supervised_columns(df)
supervised_columns

## 6. Create Player Profiles

A player profile is one row per team, player, and position. It contains the player's average performance metrics across the dataset.

In [ ]:
player_profiles = main.create_player_profiles(df, supervised_columns)
player_profiles.to_csv(main.DATA_DIR / 'player_profiles.csv', index=False)
player_profiles.head()

## 7. Create Matchup Training Data

The model needs matchup-level rows. For each match, the workflow aggregates each team's player rows into team features, then creates two perspectives: Team A vs Team B and Team B vs Team A.

In [ ]:
matchup_training_data = main.build_matchup_training_data(df)
matchup_training_data.to_csv(main.DATA_DIR / 'matchup_training_data.csv', index=False)
matchup_training_data.head()

Difference features are important. For example, `diff_avg_player_rating` means Team A's average player rating minus Team B's average player rating.

## 8. Train and Evaluate Supervised Models

The workflow trains Logistic Regression, Random Forest, and Gradient Boosting. It evaluates accuracy, classification report, confusion matrix, and log loss. Log loss is especially useful because this project predicts probabilities.

In [ ]:
X_train, X_test, y_train, y_test, training_columns = main.split_supervised_data(matchup_training_data)
models, metrics, final_model_name = main.train_supervised_models(X_train, X_test, y_train, y_test)
final_model = models[final_model_name]

main.save_object(final_model, main.MODELS_DIR / 'final_match_model.pkl')
main.save_object(training_columns, main.MODELS_DIR / 'training_columns.pkl')
main.plot_feature_importance(models, final_model_name, training_columns)

metrics

## 9. Example Custom Prediction

The prediction function validates that exactly 11 players are selected for each team, aggregates both lineups, creates matchup features, aligns the columns, and returns Team A win, draw, and Team B win probabilities.

In [ ]:
team_a, team_b, team_a_players, team_b_players = main.choose_example_lineups(player_profiles)
prediction = main.predict_custom_match(
    team_a,
    team_b,
    team_a_players,
    team_b_players,
    player_profiles,
    final_model,
    training_columns,
)

print(f"Team A: {team_a}")
print(f"Team B: {team_b}")
print(f"{team_a} win probability: {prediction['team_a_win_probability']:.1%}")
print(f"Draw probability: {prediction['draw_probability']:.1%}")
print(f"{team_b} win probability: {prediction['team_b_win_probability']:.1%}")

## 10. Player Clustering

The unsupervised section clusters player profiles. It engineers overperformance features, scales the data, tries several values of `k`, and selects a final K-Means model using silhouette score and interpretability.

In [ ]:
clustered_profiles, kmeans_model, clustering_scaler, clustering_features, k_results, cluster_summary = main.train_player_clustering(player_profiles)

clustered_profiles.to_csv(main.DATA_DIR / 'player_profiles_with_clusters.csv', index=False)
main.save_object(kmeans_model, main.MODELS_DIR / 'kmeans_model.pkl')
main.save_object(clustering_scaler, main.MODELS_DIR / 'clustering_scaler.pkl')
main.save_object(clustering_features, main.MODELS_DIR / 'clustering_features.pkl')

cluster_summary[['cluster', 'cluster_label']]

## 11. Final Artifacts

The required artifacts are saved in the `data/` and `models/` folders. The Streamlit app reads those saved files.

In [ ]:
for path in [
    main.DATA_DIR / 'player_profiles.csv',
    main.DATA_DIR / 'player_profiles_with_clusters.csv',
    main.DATA_DIR / 'matchup_training_data.csv',
    main.MODELS_DIR / 'final_match_model.pkl',
    main.MODELS_DIR / 'training_columns.pkl',
    main.MODELS_DIR / 'kmeans_model.pkl',
    main.MODELS_DIR / 'clustering_scaler.pkl',
]:
    print(path, path.exists())